# 🛒 Customer Segmentation using Spark ML

## 📌 Overview

In this project, we build an **unsupervised machine learning pipeline** using **Apache Spark** to segment customers based on their purchasing behavior.

The goal is to group customers with similar spending patterns into clusters using the **KMeans clustering algorithm**.

This project simulates a real-world business scenario where companies use customer segmentation to better understand purchasing behavior and support data-driven decision-making.

---

## 🎯 Objectives

- Load and inspect customer purchasing data  
- Prepare numerical features for clustering  
- Build a **KMeans clustering model** using Spark ML  
- Assign customers to behavioral segments  
- Analyze cluster distribution  
- Interpret customer groups based on spending patterns  

---

## 🛠️ Technologies Used

- **Apache Spark (PySpark)**
- **Spark ML**
- **KMeans Clustering**
- **Python**

---

## 📊 Dataset

The dataset contains customer purchasing information across different product categories.

Main features include:

- `Fresh_Food` — spending on fresh food products  
- `Milk` — spending on milk products  
- `Grocery` — spending on grocery products  
- `Frozen_Food` — spending on frozen food products  

💡 Since there is no target label, this is an **unsupervised learning problem**.

---

## 💡 Project Structure

The pipeline consists of the following stages:

1. Data Loading  
2. Schema Inspection  
3. Feature Engineering  
4. KMeans Model Training  
5. Cluster Analysis  
6. Business Insights  
7. Stop Spark Session  

---

💡 This project demonstrates how **Spark ML can be used for customer segmentation and business-oriented clustering analysis**.

## 1. Import + Spark Initialization

In this step, we initialize the Spark environment and import all required libraries.

Apache Spark will be used to process data and build a clustering model.

In [17]:
# Initialize Spark

import findspark
findspark.init()

from pyspark.sql import SparkSession

# Spark ML

from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler

# Create Spark session

spark = SparkSession.builder.appName("Customer_Segmentation_KMeans").getOrCreate()

print("Spark session initialized")

Spark session initialized


## 2. Data Loading

In this step, we load the customer dataset into a Spark DataFrame.

The dataset contains information about customer spending across different product categories.

In [18]:
# Load dataset

customers_df = spark.read.csv(
    "data/customers.csv",
    header=True,
    inferSchema=True
)

# Count rows

row_count = customers_df.count()

print("Total rows in dataset:", row_count)

Total rows in dataset: 440


🔍 Initial Observations
- Dataset contains 440 customer records
- All features represent spending behavior across product categories
- Data is structured and ready for analysis

## 3. Schema Inspection

Before building the clustering model, we inspect the dataset structure and data types.

This helps us:
- Understand the nature of the features  
- Ensure all columns are suitable for clustering  
- Identify any potential data issues  

In [19]:
# Inspect schema

customers_df.printSchema()

# Preview data

customers_df.show(5)

root
 |-- Fresh_Food: integer (nullable = true)
 |-- Milk: integer (nullable = true)
 |-- Grocery: integer (nullable = true)
 |-- Frozen_Food: integer (nullable = true)

+----------+----+-------+-----------+
|Fresh_Food|Milk|Grocery|Frozen_Food|
+----------+----+-------+-----------+
|     12669|9656|   7561|        214|
|      7057|9810|   9568|       1762|
|      6353|8808|   7684|       2405|
|     13265|1196|   4221|       6404|
|     22615|5410|   7198|       3915|
+----------+----+-------+-----------+
only showing top 5 rows



🔍 Initial Observations

- All columns are numerical and represent customer spending across different product categories
- Each row corresponds to an individual customer
- Spending values vary significantly across categories, indicating diverse purchasing behavior
- Some customers spend heavily in specific categories (e.g., Fresh Food or Grocery), which may form distinct clusters
- No target variable is present, confirming this is an unsupervised learning problem

💡 These characteristics make the dataset well-suited for customer segmentation using clustering techniques.

💡 Note: Feature scales differ significantly, which may impact KMeans performance and require normalization.

👉 Next step: combine all features into a single vector for clustering.

## 4. Feature Engineering

Before applying the clustering algorithm, we need to convert all input features into a single vector column.

This is required because Spark ML algorithms expect input data in vector format.

We use **VectorAssembler** to combine all numerical features into a column called **features**.

In [20]:
# Define feature columns

feature_cols = [
    "Fresh_Food",
    "Milk",
    "Grocery",
    "Frozen_Food"
]

# Create feature vector

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

# Transform dataset

customers_transformed = assembler.transform(customers_df)

# Preview features

customers_transformed.select("features").show(5, truncate=False)

+------------------------------+
|features                      |
+------------------------------+
|[12669.0,9656.0,7561.0,214.0] |
|[7057.0,9810.0,9568.0,1762.0] |
|[6353.0,8808.0,7684.0,2405.0] |
|[13265.0,1196.0,4221.0,6404.0]|
|[22615.0,5410.0,7198.0,3915.0]|
+------------------------------+
only showing top 5 rows



### 🔍 Feature Engineering Results

- All spending columns were successfully combined into a single **features** vector  
- Each vector contains four customer spending values: **Fresh Food**, **Milk**, **Grocery**, and **Frozen Food**  
- Each customer is now represented as a multidimensional data point suitable for clustering  
- The dataset is ready for KMeans model training  

💡 Next step: apply the KMeans algorithm to group customers into behavioral segments.

## 5. KMeans Model Training

In this section, we train a **KMeans clustering model** to group customers based on their purchasing behavior.

We define the number of clusters as **3**, assuming that customers can be grouped into three main behavioral segments.

KMeans will assign each customer to a cluster based on similarity in spending patterns.

In [21]:
# Define number of clusters

number_of_clusters = 3

# Create KMeans model

kmeans = KMeans(
    k=number_of_clusters,
    seed=42
)

# Train the model

model = kmeans.fit(customers_transformed)

print("KMeans model trained successfully")

KMeans model trained successfully


🔍 KMeans Training Observations

- The model was trained with k = 3 clusters, assuming three main customer segments
- Each customer is assigned to a cluster based on similarity in spending behavior
- KMeans groups customers by minimizing distance within clusters

💡 Note: The number of clusters (k) is predefined in this example.

👉 Next step: analyze cluster assignments and understand customer segments.

## 6. Cluster Analysis
In this step, we apply the trained KMeans model to assign each customer to a cluster.

This allows us to analyze how customers are distributed across different segments based on their purchasing behavior.

In [22]:
# Make predictions (assign clusters)

predictions = model.transform(customers_transformed)

# Preview results

predictions.select("features", "prediction").show(5, truncate=False)

+------------------------------+----------+
|features                      |prediction|
+------------------------------+----------+
|[12669.0,9656.0,7561.0,214.0] |0         |
|[7057.0,9810.0,9568.0,1762.0] |0         |
|[6353.0,8808.0,7684.0,2405.0] |0         |
|[13265.0,1196.0,4221.0,6404.0]|0         |
|[22615.0,5410.0,7198.0,3915.0]|1         |
+------------------------------+----------+
only showing top 5 rows



🔍 Cluster Assignment Observations

- Each customer has been assigned to a cluster based on spending similarity
- The `prediction` column represents the cluster ID for each customer
- Customers within the same cluster share similar purchasing patterns

💡 This enables segmentation of customers into distinct behavioral groups for further analysis.

👉 Next step: analyze cluster distribution and interpret customer segments.

## 7. Cluster Distribution Insights
Next, we analyze how customers are distributed across clusters.

In [23]:
# Count customers in each cluster

predictions.groupBy("prediction").count().orderBy("prediction").show()

+----------+-----+
|prediction|count|
+----------+-----+
|         0|  353|
|         1|   62|
|         2|   25|
+----------+-----+



#### Cluster Distribution Insights

- Customers were grouped into **3 distinct clusters** based on their purchasing behavior  
- Cluster distribution is uneven, indicating dominant and niche customer segments:

  - **Cluster 0 → 353 customers (~80%)** → majority group  
  - **Cluster 1 → 62 customers (~14%)** → mid-sized segment  
  - **Cluster 2 → 25 customers (~6%)** → small niche group  

- The large size of Cluster 0 suggests a **common purchasing pattern shared by most customers**  
- Smaller clusters likely represent **specialized or high-variance spending behaviors**

💡 This segmentation helps identify dominant customer groups and potential niche segments for targeted strategies.

## 8. Cluster Profiles
To better understand each segment, we examine cluster centers, which represent the average behavior of customers in each group.

In [24]:
# Show cluster centers

centers = model.clusterCenters()

for i, center in enumerate(centers):
    print(f"Cluster {i} center: {center}")

Cluster 0 center: [7853.00849858 4447.19546742 6397.88101983 2520.89518414]
Cluster 1 center: [35273.85483871  5213.91935484  5826.09677419  6027.66129032]
Cluster 2 center: [12841.6  26289.36 35155.68  3522.36]


#### Cluster Profiles

Based on cluster centers, we can interpret customer segments:

- **Cluster 0 (Majority – ~80%)**  
  - Moderate spending across all categories  
  - Balanced purchasing behavior  
  - Represents the **typical customer segment**

- **Cluster 1 (~14%)**  
  - Very high spending in **Fresh Food**  
  - Higher spending in **Frozen Food** compared with the majority group  
  - Likely represents **fresh-product-focused customers / bulk fresh food buyers**

- **Cluster 2 (~6%)**  
  - Extremely high spending in **Milk and Grocery categories**  
  - Significantly higher than other clusters  
  - Represents **high-value customers / bulk buyers**

💡 The strong differences in cluster centers indicate clear segmentation of customer behavior.

💡 Businesses can use this segmentation for:
- Targeted marketing campaigns  
- Personalized offers  
- Customer value optimization  

## 9. Detailed Prediction View

Below is a detailed (vertical) view of the clustering results for individual customers.

This format improves readability by displaying each record as a structured block, making it easier to inspect feature values and assigned cluster labels.

In [25]:
# Preview results in vertical format (for readability)

predictions.show(5, truncate=False, vertical=True)

-RECORD 0-------------------------------------
 Fresh_Food  | 12669                          
 Milk        | 9656                           
 Grocery     | 7561                           
 Frozen_Food | 214                            
 features    | [12669.0,9656.0,7561.0,214.0]  
 prediction  | 0                              
-RECORD 1-------------------------------------
 Fresh_Food  | 7057                           
 Milk        | 9810                           
 Grocery     | 9568                           
 Frozen_Food | 1762                           
 features    | [7057.0,9810.0,9568.0,1762.0]  
 prediction  | 0                              
-RECORD 2-------------------------------------
 Fresh_Food  | 6353                           
 Milk        | 8808                           
 Grocery     | 7684                           
 Frozen_Food | 2405                           
 features    | [6353.0,8808.0,7684.0,2405.0]  
 prediction  | 0                              
-RECORD 3----

🔍 Detailed View Observations

- Each record shows how individual customer spending translates into a cluster assignment
- Customers with similar spending patterns are grouped into the same segment
- This view helps validate clustering logic at the individual level

💡 It confirms that segmentation is applied consistently across different customer profiles.

## 10. Clustering Quality

To evaluate the quality of clustering, we use the **Silhouette Score**, which measures how well-separated the clusters are.

A higher score indicates that data points are well matched within their cluster and distinct from other clusters.

In [26]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator()

silhouette = evaluator.evaluate(predictions)

print("Silhouette Score:", silhouette)

Silhouette Score: 0.6963097556848293


 🔍 Clustering Quality

- The Silhouette Score is approximately 0.70, indicating well-separated and meaningful clusters  
- This suggests that customers within each cluster are similar to each other  
- At the same time, clusters are clearly distinct from one another  

💡 This confirms that the KMeans model successfully identified meaningful customer segments.
💡 This score confirms that the chosen number of clusters (k=3) is reasonable for this dataset.

## 11. Model Persistence & Export

In this step, we export clustering results and demonstrate how the trained KMeans model can be saved.

For local development, results are saved as a CSV file.
Model persistence is included as a production-style reference.

In [13]:
# Save clustering results as CSV (local development)
predictions.toPandas().to_csv("output/clustering_results.csv", index=False)
print("Clustering results saved as CSV")

Clustering results saved as CSV


In [2]:
# Save KMeans model (production-style reference)
# model.write().overwrite().save("output/kmeans_model")

# Load model example
# from pyspark.ml.clustering import KMeansModel
# loaded_model = KMeansModel.load("output/kmeans_model")

💾 Clustering results are exported as a CSV file for analysis and validation.

⚠️ Note: Model persistence in Spark may require proper Hadoop configuration.
This step is included as a production-style reference.

## 12. Inference Reference (Clustering)

In this step, we demonstrate how the trained KMeans model can assign new customers to clusters.

This reflects real-world usage of clustering models for segmenting new data.

⚠️ Note: This block may require a properly configured Spark environment.
It is provided as a production-style reference.

In [15]:
# Inference example for new customer data
# This block can be executed in a properly configured Spark environment

# new_data = spark.createDataFrame([
#     (15000, 8000, 7000, 2000)
# ], [
#     "Fresh_Food",
#     "Milk",
#     "Grocery",
#     "Frozen_Food"
# ])

# new_data_transformed = assembler.transform(new_data)

# cluster_prediction = model.transform(new_data_transformed)

# cluster_prediction.select("features", "prediction").show()

🔍 Cluster Assignment Reference

- Demonstrates how new customers can be assigned to clusters
- Shows how the trained model can be reused without retraining
- Reflects real-world customer segmentation workflows

## 13. Stop Spark Session

In the final step, we properly stop the Spark session to release allocated system resources.

This is an important best practice when working with Spark applications.

In [28]:
# Stop Spark session
spark.stop()

print("Spark session stopped successfully")

Spark session stopped successfully


---

## 14. Conclusion

This project demonstrates how clustering techniques can be applied in a real-world data engineering workflow to segment customers based on purchasing behavior.

Using Apache Spark and KMeans, we successfully:

- Built a **scalable clustering pipeline** capable of handling structured retail data  
- Transformed raw transactional data into meaningful feature representations  
- Identified **distinct customer segments** with clearly different purchasing patterns  
- Evaluated clustering performance using the **Silhouette Score (~0.70)**, confirming strong separation between clusters  

### 🔍 Key Insights

- The majority of customers belong to a **balanced spending group**, representing typical purchasing behavior  
- A smaller segment shows **focused high spending in fresh products**, indicating specialized buying habits  
- A niche group represents **high-value customers**, with significantly higher spending in specific categories  

### 💼 Business Impact

- Enables **customer segmentation for targeted marketing strategies**  
- Helps identify **high-value and niche customer groups**  
- Supports **personalized promotions and retention strategies**  
- Provides a foundation for **data-driven decision-making in retail and supply chain optimization**

### 🚀 Final Thought

This project highlights how **data engineering and machine learning pipelines** can work together to transform raw data into actionable business insights at scale.